<a href="https://colab.research.google.com/github/iman-ch/contract-conflict-checker/blob/main/01_baseline_mnli.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Notebook Overview: Create baseline model to learn generate task, get score.


## 1. Setup

### **1.1** import and install dependecies
  - transformers (pretrained models, tokenizers)
  - datasets (mnli)
  - mlflow (to record experiment settings)
  - scikit-learn

In [ ]:
import torch

In [ ]:
!pip install -q transformers datasets mlflow scikit-learn


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 2.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.1/51.1 kB 2.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.3/44.3 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 81.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.1/4.1 MB 75.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 68.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 268.7/268.7 kB 16.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 148.8/148.8 kB 14.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 228.4/228.4 kB 18.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 136.5/136.5 kB 9.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 148.7/148.7 kB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 

### **1.2** storage, experiment tracking

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
PROJ_DIR = "/content/drive/MyDrive/doc-contradiction-detector"
os.makedirs(f"{PROJ_DIR}/models", exist_ok=True)
os.makedirs(f"{PROJ_DIR}/mlruns", exist_ok=True)

Mounted at /content/drive


In [ ]:
# Opt out of the filesystem backend restriction (bug fix)
os.environ["MLFLOW_ALLOW_FILE_STORE"] = "true"

import mlflow
mlflow.set_tracking_uri(f"file:{PROJ_DIR}/mlruns")
mlflow.set_experiment("nli-distilbert")

<Experiment: artifact_location='file:///content/drive/MyDrive/doc-contradiction-detector/mlruns/647002999600830066', creation_time=1790968466273, effective_trace_archival_retention=None, experiment_id='647002999600830066', last_update_time=1790968466273, lifecycle_stage='active', name='nli-distilbert', tags={}, trace_location=None, workspace='default'>

## 2. Dataset Loading, Inspecting

### **2.1** load dataset

In [ ]:
from datasets import load_dataset, load_dataset_builder

mnli = load_dataset("nyu-mll/multi_nli")
print(mnli)

README.md:   0%|          | 0.00/8.89k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  214MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/validation_matched-00000-of-00001.p(…): reconstructing file:   0%|          |  0.00B / 4.94MB            

data/validation_matched-00000-of-00001.p(…): downloading bytes:           |  0.00B            

...

data/validation_mismatched-00000-of-0000(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/392702 [00:00<?, ? examples/s]

Generating validation_matched split:   0%|          | 0/9815 [00:00<?, ? examples/s]

Generating validation_mismatched split:   0%|          | 0/9832 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['promptID', 'pairID', 'premise', 'premise_binary_parse', 'premise_parse', 'hypothesis', 'hypothesis_binary_parse', 'hypothesis_parse', 'genre', 'label'],
        num_rows: 392702
    })
    validation_matched: Dataset({
        features: ['promptID', 'pairID', 'premise', 'premise_binary_parse', 'premise_parse', 'hypothesis', 'hypothesis_binary_parse', 'hypothesis_parse', 'genre', 'label'],
        num_rows: 9815
    })
    validation_mismatched: Dataset({
        features: ['promptID', 'pairID', 'premise', 'premise_binary_parse', 'premise_parse', 'hypothesis', 'hypothesis_binary_parse', 'hypothesis_parse', 'genre', 'label'],
        num_rows: 9832
    })
})


premise, hypothesis, genre, label
- label, genre have set values

### **2.2** inspect labels, genres

In [ ]:
from collections import Counter

label_names = mnli["train"].features["label"].names
print(label_names)
print(Counter(mnli["train"]["genre"]))


['entailment', 'neutral', 'contradiction']
Counter({'telephone': 83348, 'government': 77350, 'travel': 77350, 'fiction': 77348, 'slate': 77306})


three label types, and how often each genre appears

### **2.3** inspect examples

In [ ]:
import random
random.seed(42)
sample = random.sample(range(len(mnli["train"])), 10)

for i in sample:
  example = mnli["train"][i]
  print(f"Premise: {example['premise']}")
  print(f"Hypothesis: {example['hypothesis']}")
  print(f"Genre: {example['genre']}")
  print(f"Label: {example['label']}")
  print("-" * 30)

Premise: you know when their parents come and it's hard to get them out and a lot of parents have places to go and and things like that and it's late at night so
Hypothesis: Parents are busy and it's sometimes hard to get them out.
Genre: telephone
Label: 0
------------------------------
Premise: Where is art?
Hypothesis: Where and what is art? 
Genre: slate
Label: 1
------------------------------
Premise: Alcohol and injury, as well as brief interventions, are on the list.
Hypothesis: The list says alcohol and injury are negatives facing staff.
Genre: government
Label: 1
------------------------------
Premise: Even if gross national saving were only sufficient to replace depreciated capital, the economy could grow to some extent because replacing worn-out and used capital with new equipment tends to bring improved technology into the production process.
Hypothesis: If gross national saving was only sufficient to replace depreciated capital, the economy would still grow.
Genre: governm

sample is labelled with 2 when the premise and hypothesis contradict
- (ex. premise: temple on west side, hypothesis: temple on east side)

### **2.4** sample subsets

In [ ]:
train_sample = mnli["train"].shuffle(seed=42).select(range(20000))
val_sample = mnli["validation_matched"].shuffle(seed=42).select(range(2000))

print(train_sample)
print(val_sample)


Dataset({
    features: ['promptID', 'pairID', 'premise', 'premise_binary_parse', 'premise_parse', 'hypothesis', 'hypothesis_binary_parse', 'hypothesis_parse', 'genre', 'label'],
    num_rows: 20000
})
Dataset({
    features: ['promptID', 'pairID', 'premise', 'premise_binary_parse', 'premise_parse', 'hypothesis', 'hypothesis_binary_parse', 'hypothesis_parse', 'genre', 'label'],
    num_rows: 2000
})


## 3. Tokenization

**3.1** load tokenizer, define function

In [ ]:
import transformers
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

def tokenize(batch):
  return tokenizer(
      batch["premise"], batch["hypothesis"], # pairs
      truncation=True,
      max_length = 128,
      padding = "max_length"
  )

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

passing premise and hypothesis tells tokenizer to build pairs with [SEP] between (sep marks end of sentence)

returns:
- input ids (128 token ids, model input)
- attention mask (128 values, 1: real, 0: filler)

**3.2** apply to whole dataset

In [ ]:
train_tok = train_sample.map(tokenize, batched=True)
val_tok = val_sample.map(tokenize, batched=True)
print(tokenizer.decode(train_tok[0]["input_ids"]))

Map:   0%|          | 0/20000 [00:00<?, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

[CLS] i ' ll hurry over that part. [SEP] " i ' ll be quick with that part. " [SEP] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD]


## 4. Tensors, DataLoaders

tensor: MD array of numbers that can live on GPU and track gradients

data loaders: tool to feed data into model, takes dataset and group row into batches, stack each batch into tensor, shuffle order each epoch

**4.1** convert to tensors

In [ ]:
cols = ["input_ids", "attention_mask", "label"]
train_tok.set_format("torch", columns=cols)
val_tok.set_format("torch", columns=cols)

**4.2** create dataloaders

In [ ]:
from torch.utils.data import DataLoader

train_loader = DataLoader(train_tok, batch_size=16, shuffle=True)
val_loader = DataLoader(val_tok, batch_size=64)

batch = next(iter(train_loader))
print(batch["input_ids"].shape)   # torch.Size([16, 128])
print(batch["label"][:10])

torch.Size([16, 128])
tensor([1, 0, 1, 2, 0, 2, 1, 0, 2, 2])


collator to pad each batch only to its own longest pair, runs faster

## 5. Modelling, Optimizer, Loss

**5.1** load model onto GPU

In [ ]:
from transformers import AutoModelForSequenceClassification

device = torch.device("cuda")
model = AutoModelForSequenceClassification.from_pretrained("distilbert-base-uncased", num_labels=3).to(device)

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


**5.2** optimizer, loss function

In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5)
loss_fn = torch.nn.CrossEntropyLoss()

cross entropy loss for classification, internally applies softmax to turn logits into probabilities

## 6. Model Training

**6.1** two epoches of training

In [ ]:
num_epochs = 2

for epoch in range(num_epochs):
    # training
    model.train()
    total_loss = 0

    for batch in train_loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["label"].to(device)

        # zero
        optimizer.zero_grad()
        # fwd
        logits = model(input_ids=input_ids, attention_mask=attention_mask).logits
        # loss
        loss = loss_fn(logits, labels)
        # bkwd
        loss.backward()
        # step
        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)

    # validation
    model.eval()
    correct, total = 0, 0

    with torch.no_grad():
        for batch in val_loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["label"].to(device)

            logits = model(input_ids=input_ids, attention_mask=attention_mask).logits
            preds = logits.argmax(dim=-1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

    print(f"Epoch {epoch + 1}: train loss {avg_loss:.4f}, val accuracy {correct / total:.2%}")

Epoch 1: train loss 0.8276, val accuracy 70.30%
Epoch 2: train loss 0.5635, val accuracy 71.50%


**6.2** save model and tokenizer

In [ ]:
save_path = f"{PROJ_DIR}/models/distilbert-mnli-20k"
model.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('/content/drive/MyDrive/doc-contradiction-detector/models/distilbert-mnli-20k/tokenizer_config.json',
 '/content/drive/MyDrive/doc-contradiction-detector/models/distilbert-mnli-20k/tokenizer.json')

**6.3** test on custom sentences

In [ ]:
def predict(premise, hypothesis):
    model.eval()
    inputs = tokenizer(premise, hypothesis, return_tensors="pt").to(device)
    with torch.no_grad():
        probs = model(**inputs).logits.softmax(dim=-1)[0]
    return {name: round(p.item(), 3) for name, p in zip(label_names, probs)}

print(predict("Customers can return items within 30 days.",
              "Returns are only accepted within 14 days."))          # expect contradiction

print(predict("The meeting was moved to Friday afternoon.",
              "The meeting is happening on Friday."))               # expect entailment

print(predict("She bought a new laptop last week.",
              "She uses her laptop for work."))                     # expect neutral

{'entailment': 0.011, 'neutral': 0.06, 'contradiction': 0.93}
{'entailment': 0.965, 'neutral': 0.022, 'contradiction': 0.013}
{'entailment': 0.111, 'neutral': 0.789, 'contradiction': 0.1}
